# 🏭 Lesson 0.3: Python Memory Management — Practice Exercises

**Netsetos GenAI Course** | Module 0 — Prerequisites

6 exercises covering CPython object sizes, reference counting, circular references & generational GC, the GIL, production AI memory patterns (weakref, gc.disable), and tracemalloc leak hunting.

**Why this matters for AI engineers:** A vLLM server leaking 10MB/hour crashes after 4 days. Understanding these internals is the difference between stable deployments and 3 AM alerts.

---

## Exercise 1 (Easy): CPython Object Sizes & PyMalloc

### 📚 Theory
Every Python object has overhead: refcount (8 bytes) + type pointer (8 bytes) minimum. An empty `dict` costs 64 bytes. PyMalloc handles objects ≤512 bytes via arena allocator (256KB arenas → 4KB pools → size-class buckets). Larger objects use raw malloc. NumPy/PyTorch bypass PyMalloc entirely.

### 📋 Task
1. Measure the size of 14+ Python objects using `sys.getsizeof()`
2. Print a sorted table showing bytes and allocator (PyMalloc vs malloc)
3. Compute Python vs C overhead (Python int(0)=28 bytes, C int=4 bytes)

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 1
import sys
import numpy as np

objects = {
    "int(0)":          0,
    "int(2**30)":      2**30,
    "float(3.14)":     3.14,
    "bool(True)":      True,
    "None":            None,
    'str("")':         "",
    'str("hello")':    "hello",
    "list([])":        [],
    "list(1000)":      list(range(1000)),
    "dict({})":        {},
    "dict(100)":       {i: i for i in range(100)},
    "set()":           set(),
    "tuple()":         (),
    "np.zeros(1000)":  np.zeros(1000),
}

# TODO: print sorted table with size and allocator classification
# TODO: compute Python vs C overhead

<details><summary>✅ Solution</summary>



In [ ]:
import sys
import numpy as np

objects = {
    "int(0)":          0,
    "int(2**30)":      2**30,
    "float(3.14)":     3.14,
    "bool(True)":      True,
    "None":            None,
    'str("")':         "",
    'str("hello")':    "hello",
    "list([])":        [],
    "list(1000)":      list(range(1000)),
    "dict({})":        {},
    "dict(100)":       {i: i for i in range(100)},
    "set()":           set(),
    "tuple()":         (),
    "np.zeros(1000)":  np.zeros(1000),
}

print(f'{"Object":<22} {"Size (bytes)":>12}  {"Allocator":<10}')
print('-' * 50)
for name, obj in sorted(objects.items(), key=lambda x: sys.getsizeof(x[1])):
    size = sys.getsizeof(obj)
    allocator = "PyMalloc" if size <= 512 else "malloc"
    print(f'  {name:<20} {size:>10}  [{allocator}]')

print(f'\n--- Python vs C Overhead ---')
print(f'  Python int(0): {sys.getsizeof(0)} bytes vs C int: 4 bytes → {sys.getsizeof(0)/4:.0f}x overhead')
print(f'  Python float:  {sys.getsizeof(3.14)} bytes vs C double: 8 bytes → {sys.getsizeof(3.14)/8:.1f}x overhead')
print(f'  Python list(1000): {sys.getsizeof(list(range(1000)))} bytes (shallow — pointers only, not elements)')

</details>

---

## Exercise 2 (Easy): Reference Counting Explorer

### 📚 Theory
Every PyObject carries `ob_refcnt`. `b = a` increments it. `del b` decrements it. When it hits 0, memory is freed immediately (deterministic). `sys.getrefcount()` always reports +1 because passing the object as an argument creates a temporary reference.

Small integers (-5 to 256) are cached singletons — every use of `1` in the entire interpreter shares the same object.

### 📋 Task
1. Track refcount through: creation, alias, list append, del alias, list remove
2. Demonstrate integer caching (refcount of 1 vs 257)
3. Show that `__del__` fires immediately when refcount → 0

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 2
import sys

# Part A: Track refcounts
a = [1, 2, 3]
print(f"1. After creation:     {sys.getrefcount(a)}")  # Predict: ?

b = a
print(f"2. After b = a:        {sys.getrefcount(a)}")  # Predict: ?

# TODO: add to a container, del b, remove from container
# TODO: print refcount at each step

# Part B: Integer caching
# TODO: compare refcount of 1 vs 257

# Part C: __del__ fires immediately
class Tracked:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"  → {self.name} freed (refcount hit 0)")

# TODO: create and del a Tracked object

<details><summary>✅ Solution</summary>



In [ ]:
import sys

# Part A: Track refcounts through operations
print('--- Part A: Reference Count Tracking ---')
a = [1, 2, 3]
print(f"1. After creation:     {sys.getrefcount(a)}")  # 2 (var + getrefcount arg)

b = a  # alias
print(f"2. After b = a:        {sys.getrefcount(a)}")  # 3

container = [a]  # stored in a list
print(f"3. After list append:  {sys.getrefcount(a)}")  # 4

del b
print(f"4. After del b:        {sys.getrefcount(a)}")  # 3

container.pop()
print(f"5. After list remove:  {sys.getrefcount(a)}")  # 2

# Part B: Integer caching
print('\n--- Part B: Integer Caching ---')
print(f"refcount(1):   {sys.getrefcount(1):>6}  (cached singleton — shared across interpreter)")
print(f"refcount(42):  {sys.getrefcount(42):>6}  (also cached)")
print(f"refcount(257): {sys.getrefcount(257):>6}  (NOT cached — created fresh each time)")
print(f"refcount(999): {sys.getrefcount(999):>6}  (NOT cached)")

# Part C: __del__ fires immediately when refcount → 0
print('\n--- Part C: Immediate __del__ ---')
class Tracked:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"  → {self.name} freed (refcount hit 0)")

print("Creating object...")
obj = Tracked("test-object")
print("Deleting object...")
del obj  # __del__ fires IMMEDIATELY, no gc.collect() needed
print("Done — __del__ already fired above (deterministic!)")

</details>

---

## Exercise 3 (Medium): Circular Reference Detector & Generational GC

### 📚 Theory
Refcounting can't detect cycles (A→B→A). The generational GC tracks container objects across 3 generations with thresholds `(700, 10, 10)`:
- **Gen 0:** Collected every ~700 net allocations (most objects die young)
- **Gen 1:** Every 10 Gen-0 collections
- **Gen 2:** Every 10 Gen-1 collections (long-lived objects)

`gc.collect(generation)` forces collection of a specific generation.

### 📋 Task
1. Create two objects pointing to each other (A→B→A cycle)
2. Delete both — prove `__del__` does NOT fire (refcount > 0)
3. `gc.collect()` — now `__del__` fires
4. Print `gc.get_threshold()` and `gc.get_count()`
5. Create cycles in a loop, collect per-generation

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 3
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __del__(self):
        print(f"  → {self.name} collected by GC")

# Part A: Create a cycle
a = Node("A")
b = Node("B")
a.partner = b  # A → B
b.partner = a  # B → A (cycle!)

print("Before del:")
del a, b
print("After del (nothing freed — cycle keeps refcount > 0)")

# TODO: gc.collect() to break the cycle

# Part B: Inspect GC state
# TODO: print gc.get_threshold() and gc.get_count()

# Part C: Per-generation collection
# TODO: create multiple cycles, collect gen 0 vs gen 1

<details><summary>✅ Solution</summary>



In [ ]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __del__(self):
        print(f"  → {self.name} collected by GC")

# Part A: Circular reference
print('--- Part A: Circular Reference ---')
gc.collect()  # Clean slate

a = Node("A")
b = Node("B")
a.partner = b  # A → B
b.partner = a  # B → A (cycle!)

print("Before del:")
del a, b
print("After del (nothing freed — cycle keeps refcount > 0)")

collected = gc.collect()
print(f"gc.collect() → freed {collected} objects")
print()

# Part B: Inspect GC state
print('--- Part B: GC State ---')
print(f"Thresholds: {gc.get_threshold()}")  # (700, 10, 10)
print(f"Counts:     {gc.get_count()}")       # (gen0_allocs, gen1_count, gen2_count)
print()

# Part C: Per-generation collection
print('--- Part C: Per-Generation Collection ---')

class SilentNode:
    """Node without __del__ to avoid noisy output."""
    def __init__(self):
        self.partner = None

# Create 10 cycles
for i in range(10):
    x = SilentNode()
    y = SilentNode()
    x.partner = y
    y.partner = x
    del x, y

print(f"Before collection — Counts: {gc.get_count()}")

collected_gen0 = gc.collect(0)
print(f"gc.collect(0) [Gen 0 only]: freed {collected_gen0} objects")
print(f"After Gen 0 — Counts: {gc.get_count()}")

collected_gen1 = gc.collect(1)
print(f"gc.collect(1) [Gen 0+1]:   freed {collected_gen1} objects")
print(f"After Gen 1 — Counts: {gc.get_count()}")

collected_full = gc.collect(2)
print(f"gc.collect(2) [Full]:      freed {collected_full} objects")

</details>

---

## Exercise 4 (Medium): GIL Impact Benchmark

### 📚 Theory
The GIL allows only one thread to execute Python bytecode at a time. Threading does NOT speed up CPU-bound Python. But: NumPy/PyTorch release the GIL during C-level computation, so threading DOES help for matrix ops. For I/O-bound tasks (LLM API calls), async/threading both work.

### 📋 Task
1. Write a CPU-bound function (count to 5M in a Python loop)
2. Write a simulated I/O-bound function (`time.sleep(1)`)
3. Run 4 copies: sequential vs threaded. Time each.
4. Print comparison table showing where GIL blocks

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 4
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_work(n=5_000_000):
    """CPU-bound: pure Python loop (GIL blocks parallelism)."""
    total = 0
    for i in range(n):
        total += i
    return total

def io_work(seconds=1.0):
    """I/O-bound: simulated API call (GIL released during sleep)."""
    time.sleep(seconds)
    return "done"

NUM_TASKS = 4

# TODO: sequential CPU, threaded CPU — compare times
# TODO: sequential I/O, threaded I/O — compare times
# TODO: print comparison table

<details><summary>✅ Solution</summary>



In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_work(n=5_000_000):
    """CPU-bound: pure Python loop (GIL blocks parallelism)."""
    total = 0
    for i in range(n):
        total += i
    return total

def io_work(seconds=1.0):
    """I/O-bound: simulated API call (GIL released during sleep)."""
    time.sleep(seconds)
    return "done"

NUM_TASKS = 4

# --- CPU-bound benchmark ---
print('--- CPU-bound (count to 5M × 4) ---')

start = time.time()
for _ in range(NUM_TASKS):
    cpu_work()
seq_cpu = time.time() - start
print(f'  Sequential:   {seq_cpu:.2f}s')

start = time.time()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as ex:
    list(ex.map(lambda _: cpu_work(), range(NUM_TASKS)))
thr_cpu = time.time() - start
print(f'  Threaded (4): {thr_cpu:.2f}s')
print(f'  Speedup:      {seq_cpu/thr_cpu:.2f}x  (≈1x → GIL blocks!)\n')

# --- I/O-bound benchmark ---
print('--- I/O-bound (sleep 1s × 4) ---')

start = time.time()
for _ in range(NUM_TASKS):
    io_work()
seq_io = time.time() - start
print(f'  Sequential:   {seq_io:.2f}s')

start = time.time()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as ex:
    list(ex.map(lambda _: io_work(), range(NUM_TASKS)))
thr_io = time.time() - start
print(f'  Threaded (4): {thr_io:.2f}s')
print(f'  Speedup:      {seq_io/thr_io:.2f}x  (≈4x → GIL released!)\n')

# --- Summary ---
print('--- Verdict ---')
print(f'  CPU-bound: threading gives {seq_cpu/thr_cpu:.1f}x (GIL blocks → use multiprocessing)')
print(f'  I/O-bound: threading gives {seq_io/thr_io:.1f}x (GIL released → threading/async work)')
print(f'  LLM API calls = I/O-bound → use asyncio.gather() (Lesson 0.1!)')

</details>

---

## Exercise 5 (Hard): Production AI Memory Patterns

### 📚 Theory
**Pattern 1: WeakRef Cache** — `weakref.WeakValueDictionary` holds weak references. Entries auto-evict when no strong reference exists. Perfect for embedding caches that shouldn't grow unbounded.

**Pattern 2: gc.disable during inference** — Prevents random 10-50ms GC pauses during latency-sensitive serving. Re-enable + collect between batches.

**Pattern 3: Explicit cleanup** — `del tensor; gc.collect()` between epochs frees fragmented memory.

### 📋 Task
1. Build a `WeakRefEmbeddingCache` — auto-evicts when strong refs are deleted
2. Compare: regular dict (grows forever) vs weakref dict (self-cleans)
3. Implement gc.disable/enable pattern with timing
4. Simulate epoch cleanup loop with memory tracking

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 5
import gc, weakref, time
import numpy as np

class Embedding:
    """Mock embedding object (1536-dim vector, ~6KB each)."""
    def __init__(self, text):
        self.text = text
        self.vector = np.random.randn(1536).astype(np.float32)

class WeakRefEmbeddingCache:
    def __init__(self):
        self._cache = weakref.WeakValueDictionary()
        self.hits = 0
        self.misses = 0

    def get(self, text):
        # TODO: check cache → hit or miss
        pass

    def __len__(self):
        return len(self._cache)

# TODO: store 5 embeddings, keep strong refs to 2, del the rest
# TODO: gc.collect(), show cache shrank

# TODO: gc.disable/enable pattern with timing

# TODO: epoch cleanup loop

<details><summary>✅ Solution</summary>



In [ ]:
import gc, weakref, time, tracemalloc
import numpy as np

class Embedding:
    """Mock embedding object (1536-dim vector, ~6KB each)."""
    def __init__(self, text):
        self.text = text
        self.vector = np.random.randn(1536).astype(np.float32)

# --- Pattern 1: WeakRef Embedding Cache ---
print('--- Pattern 1: WeakRef Cache ---')

class WeakRefEmbeddingCache:
    def __init__(self):
        self._cache = weakref.WeakValueDictionary()
        self.hits = 0
        self.misses = 0

    def get(self, text):
        if text in self._cache:
            self.hits += 1
            return self._cache[text]
        self.misses += 1
        emb = Embedding(text)
        self._cache[text] = emb
        return emb

    def stats(self):
        return f'size={len(self._cache)}, hits={self.hits}, misses={self.misses}'

cache = WeakRefEmbeddingCache()

# Store 5 embeddings, keep strong refs to only 2
keep1 = cache.get("hello")
keep2 = cache.get("world")
_ = cache.get("temp1")  # no strong ref kept
_ = cache.get("temp2")  # no strong ref kept
_ = cache.get("temp3")  # no strong ref kept
_ = None  # release the last assignment

print(f'After storing 5: {cache.stats()}')

gc.collect()
print(f'After gc.collect: {cache.stats()}')
print('(temp entries auto-evicted — only strong refs survive!)\n')

# --- Pattern 2: gc.disable during inference ---
print('--- Pattern 2: gc.disable During Inference ---')

def simulated_inference():
    """Simulate a batch of matrix operations."""
    for _ in range(100):
        a = np.random.randn(256, 256)
        b = np.random.randn(256, 256)
        c = a @ b
    return c

# With GC enabled
times_gc_on = []
for _ in range(5):
    start = time.time()
    simulated_inference()
    times_gc_on.append((time.time() - start) * 1000)

# With GC disabled
times_gc_off = []
for _ in range(5):
    gc.disable()
    start = time.time()
    simulated_inference()
    elapsed = (time.time() - start) * 1000
    gc.enable()
    gc.collect()
    times_gc_off.append(elapsed)

avg_on = sum(times_gc_on) / len(times_gc_on)
avg_off = sum(times_gc_off) / len(times_gc_off)
print(f'  GC enabled:  {avg_on:.1f}ms avg')
print(f'  GC disabled: {avg_off:.1f}ms avg')
print(f'  (gc.disable prevents random GC pauses during serving)\n')

# --- Pattern 3: Epoch Cleanup ---
print('--- Pattern 3: Epoch Cleanup ---')
tracemalloc.start()

for epoch in range(3):
    # Simulate allocating a large training batch
    big_batch = np.random.randn(10000, 128).astype(np.float32)
    current, peak = tracemalloc.get_traced_memory()
    print(f'  Epoch {epoch+1}: allocated {current/1024/1024:.1f}MB (peak: {peak/1024/1024:.1f}MB)')

    # Explicit cleanup
    del big_batch
    gc.collect()
    current, peak = tracemalloc.get_traced_memory()
    print(f'           after cleanup: {current/1024/1024:.1f}MB (peak: {peak/1024/1024:.1f}MB)')

tracemalloc.stop()
print('\nExplicit del + gc.collect() between epochs keeps memory flat.')

</details>

---

## Exercise 6 (Hard): Memory Leak Hunter with tracemalloc

### 📚 Theory
`tracemalloc` tracks every allocation's source file and line number. Snapshot A → run leaky code → Snapshot B → diff reveals exactly where memory grew. This is how you debug "why is my vLLM server using 12GB after 4 hours?"

### 📋 Task
1. Start tracemalloc, take snapshot A
2. Run a function with an intentional leak (appending to a global list)
3. Take snapshot B, diff A→B — identify the leaking line
4. Fix the leak (clear list or use bounded deque)
5. Take snapshot C, diff B→C — prove the fix works
6. Print peak and current memory

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 6
import tracemalloc
import gc
import numpy as np

tracemalloc.start()
snap_a = tracemalloc.take_snapshot()

# Leaky function
leaked_cache = []

def process_batch(batch_id):
    embedding = np.random.randn(1536).astype(np.float32)
    leaked_cache.append(embedding)  # LEAK!
    return f"Batch {batch_id} done"

# Run 500 batches
for i in range(500):
    process_batch(i)

snap_b = tracemalloc.take_snapshot()

# TODO: diff snap_a → snap_b, print top 3 growers
# TODO: fix the leak
# TODO: take snap_c, diff snap_b → snap_c, prove fix
# TODO: print current and peak memory

<details><summary>✅ Solution</summary>



In [ ]:
import tracemalloc
import gc
import numpy as np
from collections import deque

tracemalloc.start()

# --- Snapshot A (baseline) ---
snap_a = tracemalloc.take_snapshot()

# --- Leaky function ---
leaked_cache = []  # BUG: global list that grows forever

def process_batch(batch_id):
    """Simulates processing that leaks memory."""
    embedding = np.random.randn(1536).astype(np.float32)
    leaked_cache.append(embedding)  # LEAK: never cleaned up!
    return f"Batch {batch_id} processed"

# Run 500 "batches"
for i in range(500):
    process_batch(i)

# --- Snapshot B (after leak) ---
snap_b = tracemalloc.take_snapshot()

print('=== LEAK DETECTION: Snapshot A → B ===')
print('Top memory growers:')
for stat in snap_b.compare_to(snap_a, 'lineno')[:3]:
    print(f'  {stat}')

current_b, peak_b = tracemalloc.get_traced_memory()
print(f'\nCurrent: {current_b/1024:.1f} KB, Peak: {peak_b/1024:.1f} KB')
print(f'Leaked cache size: {len(leaked_cache)} entries\n')

# --- Fix the leak ---
print('=== FIXING LEAK ===')
leaked_cache.clear()
gc.collect()
print('leaked_cache.clear() + gc.collect()\n')

# --- Snapshot C (after fix) ---
snap_c = tracemalloc.take_snapshot()

print('=== VERIFICATION: Snapshot B → C ===')
print('Top memory changes (negative = freed):')
for stat in snap_c.compare_to(snap_b, 'lineno')[:3]:
    print(f'  {stat}')

current_c, peak_c = tracemalloc.get_traced_memory()
print(f'\nCurrent: {current_c/1024:.1f} KB, Peak: {peak_c/1024:.1f} KB')
print(f'Memory freed: {(current_b - current_c)/1024:.1f} KB\n')

# --- Bonus: Bounded cache with deque ---
print('=== BONUS: Bounded Cache with deque(maxlen=100) ===')
bounded_cache = deque(maxlen=100)
for i in range(500):
    bounded_cache.append(np.random.randn(1536).astype(np.float32))
print(f'After 500 inserts, cache size: {len(bounded_cache)} (bounded!)')
print(f'Memory stays flat no matter how many batches you process.')

tracemalloc.stop()

</details>

---

## 🎉 Done!

You've mastered all 6 Python memory management concepts for production AI:
- **CPython Architecture** — object sizes, PyMalloc vs malloc
- **Reference Counting** — deterministic, immediate, predictable
- **Circular References & GC** — generational collector, thresholds, per-gen collection
- **The GIL** — threading vs multiprocessing vs async, when each works
- **Production Patterns** — weakref caches, gc.disable, explicit cleanup
- **tracemalloc** — snapshot diffing to find and fix memory leaks

These are the exact skills tested in senior Python interviews at Indian IT-services companies — and the patterns you'll use daily in production AI deployments. Next: **Module 1.**